# Train one backbone

One notebook, five backbones. Change `BACKBONE` and run the notebook again. Do not change the label order or the preprocess output between runs, or the ensemble in the next notebook will refuse to average them.

Stage 1 freezes the backbone and trains the new sigmoid head for 2 epochs at 1e-3. Stage 2 unfreezes the top 30% of backbone layers, keeps every BatchNorm layer frozen, and trains at 1e-4 with cosine decay. The loss is weighted binary cross-entropy from the training-split label frequencies.

Each epoch writes a checkpoint. If the session stops after a checkpoint has been saved, run the same backbone again with the same output directory. The notebook continues from the latest epoch. New epochs are not started once the session is about 4.5 hours in, so prediction files can still be written.

Set `WEIGHTS = None` to skip the ImageNet download. On a GPU run leave `WEIGHTS = "imagenet"`.


In [ ]:
import os

# EfficientNetB0, DenseNet201, ResNet50, InceptionV3, Xception
BACKBONE = "ResNet50"
SEED = 42
EPOCHS = 6  # stage 2. Stage 1 uses STAGE1_EPOCHS.
STAGE1_EPOCHS = 2
WEIGHTS = "imagenet"  # set to None to skip ImageNet initialization
BATCH_SIZE = None  # None picks a size for this backbone
DATA_DIR = os.environ.get("DATA_DIR", "data/nih")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "data/output")

if os.environ.get("CXR_BACKBONE"):
    BACKBONE = os.environ["CXR_BACKBONE"]
if os.environ.get("CXR_SEED"):
    SEED = int(os.environ["CXR_SEED"])
if os.environ.get("CXR_EPOCHS"):
    EPOCHS = int(os.environ["CXR_EPOCHS"])
if "CXR_WEIGHTS" in os.environ:
    raw = os.environ["CXR_WEIGHTS"].strip()
    WEIGHTS = None if raw.lower() in {"none", ""} else raw
print("BACKBONE", BACKBONE, "SEED", SEED, "STAGE1", STAGE1_EPOCHS, "STAGE2", EPOCHS)
print("WEIGHTS", WEIGHTS)
print("DATA_DIR", DATA_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)


In [ ]:
import sys
from pathlib import Path

def add_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *list(here.parents)[:4]]
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "src" / "__init__.py").is_file() and (resolved / "src" / "paths.py").is_file():
            root = str(resolved)
            if root not in sys.path:
                sys.path.insert(0, root)
            print("Using code at", resolved)
            return resolved
    raise FileNotFoundError(
        "Could not find src/paths.py. Run this notebook from the project directory."
    )

add_project_root()


In [ ]:
from src.data import preprocess_means

means = preprocess_means()
for name, value in means.items():
    print(f"{name:16s} preprocess mean {value:.4f}")


In [ ]:
from src.train import run_training

result = run_training(
    backbone=BACKBONE,
    seed=SEED,
    stage1_epochs=STAGE1_EPOCHS,
    stage2_epochs=EPOCHS,
    weights=WEIGHTS,
    batch_size=BATCH_SIZE,
    out_dir=OUTPUT_DIR,
    preprocessed_dir=OUTPUT_DIR,
)
print(result)
